# Hands-on Week 10

## Building a Vector Search Engine

In this notebook, we build the retrieval component of a Retrieval-Augmented Generation (RAG) system from scratch.

The goal is to understand how semantic search works before adding an LLM.

### Learning Objectives

By the end of this notebook, you should be able to:

- explain the difference between keyword search and semantic search
- create sentence embeddings for short documents
- compute cosine similarity
- retrieve relevant documents using vector similarity
- build a FAISS index
- compare lexical and semantic retrieval behavior
- understand why retrieval quality is central for RAG

---

In [ ]:
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

try:
    import faiss
    FAISS_AVAILABLE = True
except ImportError:
    FAISS_AVAILABLE = False

def TODO(todo: str = "Fill the blank"):
    raise ValueError(todo)

random.seed(42)
np.random.seed(42)

## 1. A Small Document Corpus

We start with a small collection of short documents.

Each document is similar to a chunk in a real RAG system.  
In practice, these chunks might come from PDFs, invoices, contracts, reports, or webpages.

Here we use short artificial but realistic documents so that the retrieval behavior is easy to inspect.

In [ ]:
documents = [
    {"id": "doc_01", "category": "invoice", "text": "Invoice INV-2024-001 was issued by Northwind Supplies to ACME GmbH. The total amount is 1,245.90 EUR and the due date is 2024-05-30."},
    {"id": "doc_02", "category": "invoice", "text": "Receipt from City Market. The customer paid 54.80 EUR by credit card on 2024-04-12. The merchant address is Hamburg Central Station."},
    {"id": "doc_03", "category": "contract", "text": "The service agreement between DataVision Ltd. and Blue Ocean GmbH starts on 2024-06-01 and ends on 2025-05-31."},
    {"id": "doc_04", "category": "contract", "text": "The contract includes a confidentiality clause. Both parties agree not to disclose sensitive business information to third parties."},
    {"id": "doc_05", "category": "resume", "text": "Maria Schmidt is a data scientist with experience in Python, machine learning, and natural language processing."},
    {"id": "doc_06", "category": "resume", "text": "John Miller worked as a backend engineer and has strong experience with Docker, Kubernetes, and cloud infrastructure."},
    {"id": "doc_07", "category": "medical", "text": "The patient reported fever, headache, and fatigue. The doctor recommended rest and increased fluid intake."},
    {"id": "doc_08", "category": "medical", "text": "The lab report shows elevated glucose levels. A follow-up appointment with an endocrinologist was recommended."},
    {"id": "doc_09", "category": "research", "text": "The paper introduces a transformer-based model for document information extraction from scanned invoices."},
    {"id": "doc_10", "category": "research", "text": "The study compares BM25 and dense retrieval for question answering over scientific documents."},
    {"id": "doc_11", "category": "travel", "text": "The train from Hamburg to Copenhagen departs at 08:56 and arrives at 13:34. Seat reservation is recommended."},
    {"id": "doc_12", "category": "travel", "text": "The hotel booking in Paris includes breakfast and free cancellation until 24 hours before arrival."},
    {"id": "doc_13", "category": "invoice", "text": "Invoice INV-2024-089 was issued to GreenTech AG. The invoice total is 3,780.00 EUR including VAT."},
    {"id": "doc_14", "category": "contract", "text": "Payment must be made within 14 days after receipt of the invoice. Late payments may result in additional fees."},
    {"id": "doc_15", "category": "research", "text": "Retrieval-Augmented Generation combines document retrieval with large language models to produce grounded answers."},
    {"id": "doc_16", "category": "medical", "text": "The prescription contains ibuprofen 400 mg. The patient should take one tablet after meals if needed."},
    {"id": "doc_17", "category": "resume", "text": "Anna Weber has experience in data engineering, SQL databases, Apache Airflow, and ETL pipelines."},
    {"id": "doc_18", "category": "travel", "text": "The flight to Berlin was delayed by two hours due to severe weather conditions."},
    {"id": "doc_19", "category": "invoice", "text": "The payment reference for this invoice is RF-99821. Please include the reference when making a bank transfer."},
    {"id": "doc_20", "category": "research", "text": "Vector databases store embeddings and allow nearest-neighbor search over large document collections."}
]

df = pd.DataFrame(documents)
df

## 2. Keyword Search Baseline

Before using embeddings, we implement a very simple keyword-based search.

This is similar in spirit to classical information retrieval:

- tokenize the query
- tokenize each document
- count how many query words appear in the document

This approach is simple and interpretable, but it struggles with synonyms and semantic similarity.

In [ ]:
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

def keyword_search(query, documents, top_k=5):
    query_terms = set(tokenize(query))
    results = []

    for doc in documents:
        doc_terms = set(tokenize(doc["text"]))
        overlap = len(query_terms.intersection(doc_terms))
        results.append({
            "id": doc["id"],
            "category": doc["category"],
            "score": overlap,
            "text": doc["text"]
        })

    results = sorted(results, key=lambda x: x["score"], reverse=True)
    return pd.DataFrame(results[:top_k])

keyword_search("invoice total amount", documents, top_k=5)

In [ ]:
keyword_search("How much money needs to be paid?", documents, top_k=5)

## 3. Dense Embeddings

Dense retrieval represents each document as a vector.

The key idea:

> Meaning becomes geometry.

Documents with similar meaning should have nearby vectors in embedding space.

We use a SentenceTransformer model to create embeddings.

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [doc["text"] for doc in documents]
embeddings = model.encode(texts, convert_to_numpy=True)

print("Embedding matrix shape:", embeddings.shape)
print("Number of documents:", embeddings.shape[0])
print("Embedding dimension:", embeddings.shape[1])

## 4. Inspect One Embedding

Each document is now represented by a dense vector.

The model we use produces a 384-dimensional vector for each text.

In [ ]:
doc_index = 0

print("Document:")
print(texts[doc_index])
print()
print("First 10 embedding dimensions:")
print(embeddings[doc_index][:10])

## 5. Cosine Similarity

To search semantically, we compare vectors.

A common similarity function is cosine similarity:

\[
\cos(\theta)=\frac{x \cdot y}{\|x\|\|y\|}
\]

High cosine similarity means two vectors point in a similar direction.

In [ ]:
def semantic_search(query, documents, embeddings, model, top_k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    similarities = cosine_similarity(query_embedding, embeddings)[0]

    results = []
    for i, doc in enumerate(documents):
        results.append({
            "id": doc["id"],
            "category": doc["category"],
            "score": similarities[i],
            "text": doc["text"]
        })

    results = sorted(results, key=lambda x: x["score"], reverse=True)
    return pd.DataFrame(results[:top_k])

semantic_search("How much money needs to be paid?", documents, embeddings, model, top_k=5)

## 6. Keyword Search vs Semantic Search

Now compare the two approaches.

Try a query where the exact keyword does not appear in the best matching document.

In [ ]:
query = "Which document talks about semantic search over documents?"

print("Keyword Search")
display(keyword_search(query, documents, top_k=5))

print("Semantic Search")
display(semantic_search(query, documents, embeddings, model, top_k=5))

## 7. Visualizing Embeddings with PCA

Embedding vectors are high-dimensional.

To visualize them, we reduce them to two dimensions using PCA.

This is only for visualization; the real search happens in the original embedding space.

In [ ]:
pca = PCA(n_components=2, random_state=42)
embeddings_2d = pca.fit_transform(embeddings)

plot_df = pd.DataFrame({
    "x": embeddings_2d[:, 0],
    "y": embeddings_2d[:, 1],
    "category": [doc["category"] for doc in documents],
    "id": [doc["id"] for doc in documents]
})

plt.figure(figsize=(9, 6))

for category in sorted(plot_df["category"].unique()):
    subset = plot_df[plot_df["category"] == category]
    plt.scatter(subset["x"], subset["y"], label=category, s=80)

for _, row in plot_df.iterrows():
    plt.text(row["x"] + 0.01, row["y"] + 0.01, row["id"], fontsize=8)

plt.title("Document Embeddings Visualized with PCA")
plt.xlabel("PCA dimension 1")
plt.ylabel("PCA dimension 2")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 8. Building a FAISS Index

For small datasets, we can compute cosine similarity against every document.

For large datasets, this becomes expensive.

FAISS is a library for efficient nearest-neighbor search over vectors.

Here we build a simple FAISS index.

In [ ]:
if not FAISS_AVAILABLE:
    print("FAISS is not installed. Install it with: pip install faiss-cpu")
else:
    normalized_embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

    embedding_dim = normalized_embeddings.shape[1]
    index = faiss.IndexFlatL2(embedding_dim)
    index.add(normalized_embeddings.astype("float32"))

    print("Number of vectors in index:", index.ntotal)

In [ ]:
def faiss_search(query, documents, model, index, top_k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    query_embedding = query_embedding / np.linalg.norm(query_embedding, axis=1, keepdims=True)

    distances, indices = index.search(query_embedding.astype("float32"), top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        doc = documents[idx]
        results.append({
            "rank": rank + 1,
            "id": doc["id"],
            "category": doc["category"],
            "distance": distances[0][rank],
            "text": doc["text"]
        })

    return pd.DataFrame(results)

if FAISS_AVAILABLE:
    faiss_search("Find documents about vector search and retrieval", documents, model, index, top_k=5)

## 9. Exercise 1 — Try Your Own Queries

Try at least three different queries.

Examples:

- `Which document mentions a due date?`
- `Find documents about medical treatment.`
- `Which document describes a train journey?`
- `Find information about machine learning for documents.`

Compare keyword search and semantic search.

In [ ]:
query = "Find information about machine learning for documents"

print("Keyword Search")
display(keyword_search(query, documents, top_k=5))

print("Semantic Search")
display(semantic_search(query, documents, embeddings, model, top_k=5))

if FAISS_AVAILABLE:
    print("FAISS Search")
    display(faiss_search(query, documents, model, index, top_k=5))

## 10. Exercise 2 — Add New Documents

Add at least two new documents to the corpus.

Then recompute the embeddings and repeat semantic search.

In [ ]:
new_documents = [
    {
        "id": "doc_21",
        "category": "custom",
        "text": "TODO: Add your own document text here."
    },
    {
        "id": "doc_22",
        "category": "custom",
        "text": "TODO: Add another document text here."
    }
]

# TODO:
# 1. Add the new documents to the original documents list.
# 2. Recompute embeddings.
# 3. Run semantic search again.

# extended_documents = documents + new_documents
# extended_texts = [doc["text"] for doc in extended_documents]
# extended_embeddings = model.encode(extended_texts, convert_to_numpy=True)
# semantic_search("your query here", extended_documents, extended_embeddings, model, top_k=5)

## 11. Exercise 3 — Inspect Retrieval Failures

Semantic search is powerful, but not perfect.

Try to find a query where the top result is not what you expected.

Questions:

1. What was your query?
2. Which document was retrieved?
3. Why might the embedding model have confused the meaning?
4. How could this be improved?

In [ ]:
query = "TODO: Write a difficult or ambiguous query here"

# display(semantic_search(query, documents, embeddings, model, top_k=5))

## 12. Exercise 4 — From Retrieval to RAG

In RAG, the retrieval result is passed to an LLM as context.

Here we only build the retrieval side.

Complete the prompt template below.

In [ ]:
query = "What is the total amount of invoice INV-2024-001?"

retrieved = semantic_search(query, documents, embeddings, model, top_k=3)

context = "\n\n".join(retrieved["text"].tolist())

prompt = f"""
You are an information extraction system.

Use only the context below to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""

print(prompt)

## Reflection Questions

Answer briefly:

1. Why does keyword search fail for some queries?
2. What does an embedding vector represent?
3. Why is cosine similarity useful for semantic search?
4. Why do we need vector indexes such as FAISS?
5. What could go wrong if retrieval returns the wrong document?
6. How is this notebook connected to RAG?
7. Why is retrieval quality important for Information Extraction?

## Summary

In this notebook, we built the retrieval component of RAG.

We covered:

- keyword search
- dense embeddings
- cosine similarity
- PCA visualization
- semantic search
- FAISS vector search
- retrieval-based prompt construction

Main takeaway:

> RAG depends on retrieval quality. If the retriever returns the wrong context, the LLM is likely to produce a wrong or hallucinated answer.

In the next notebook, we will add an LLM and perform RAG-based Information Extraction.